# Livrable 2 - Passage à Spark

Ce notebook va reproduire la pipeline du nettoyage des données d'un csv du premier livrables, en l'adaptant avec un notebook Spark



## Création de la SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                    .appName("DataImme_Livrable2")
                    .getOrCreate()
)

# Affichage des informations + génération d'un lien vers spark
spark

Définition d'un schéma pour les données de notre fichier

In [5]:
from pyspark.sql.functions import col

# Lecture du fichier csv compréssé

df_brut = spark.read.csv(
    "../data/raw/dvf_59_2024.csv.gz",
    header = True
)

df_spark = df_brut.select(
    col("id_mutation").cast("string"),
    col("date_mutation").cast("date"),
    col("valeur_fonciere").cast("double"),
    col("code_postal").cast("string"),
    col("code_commune").cast("string"),
    col("nom_commune").cast("string"),
    col("code_departement").cast("string"),
    col("type_local").cast("string"),
    col("surface_reelle_bati").cast("double"),
    col("nombre_pieces_principales").cast("double"),
    col("longitude").cast("double"),
    col("latitude").cast("double")
)

df_spark.show(5)

+-----------+-------------+---------------+-----------+------------+--------------+----------------+----------+-------------------+-------------------------+---------+---------+
|id_mutation|date_mutation|valeur_fonciere|code_postal|code_commune|   nom_commune|code_departement|type_local|surface_reelle_bati|nombre_pieces_principales|longitude| latitude|
+-----------+-------------+---------------+-----------+------------+--------------+----------------+----------+-------------------+-------------------------+---------+---------+
|2024-680228|   2024-01-03|       420000.0|      59710|       59466|  Pont-à-Marcq|              59|    Maison|              122.0|                      4.0|  3.10731|  50.5241|
|2024-680229|   2024-01-04|       476700.0|      59390|       59367|Lys-lez-Lannoy|              59|    Maison|              203.0|                      7.0| 3.213318|50.674598|
|2024-680230|   2024-01-03|       153000.0|      59187|       59170|         Dechy|              59|    Maison

## Nettoyage des données / enregistrements

In [9]:
from pyspark.sql.functions import col, round


# suppression des enregistrements en doublons
df_propre = df_spark.dropDuplicates(["id_mutation"])

# suppresion des lignes sans prix ou surface
df_propre = df_propre.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])

# Créationd e la colonne prix m2
df_propre = df_propre.withColumn(
    "prix_m2",
    col("valeur_fonciere") / col("surface_reelle_bati")
)

# Suppression des enregistrements avec un prix au m2 irréels
df_propre = df_propre.filter((col("prix_m2") >= 100) & (col("prix_m2") <= 20000))
df_propre = df_propre.withColumn("prix_m2", round(col("prix_m2"), 2))

print("Enregistrements nettoyé")
df_propre.show(5)

Enregistrements nettoyé
+-----------+-------------+---------------+-----------+------------+--------------------+----------------+-----------+-------------------+-------------------------+---------+---------+-------+
|id_mutation|date_mutation|valeur_fonciere|code_postal|code_commune|         nom_commune|code_departement| type_local|surface_reelle_bati|nombre_pieces_principales|longitude| latitude|prix_m2|
+-----------+-------------+---------------+-----------+------------+--------------------+----------------+-----------+-------------------+-------------------------+---------+---------+-------+
|2024-680228|   2024-01-03|       420000.0|      59710|       59466|        Pont-à-Marcq|              59|     Maison|              122.0|                      4.0|  3.10731|  50.5241|3442.62|
|2024-680230|   2024-01-03|       153000.0|      59187|       59170|               Dechy|              59|     Maison|              115.0|                      6.0| 3.126817|50.345947|1330.43|
|2024-68023

Lorsque l'on nettoie les données avec filter ou withColumn, Spark ne calcule rien tout de suite. Il se contente de prendre des notes et de préparer une "recette" (Lazy Evaluation)

### Calcul du prix moyen au m2 par commune et par type de bien

In [10]:
from pyspark.sql.functions import avg

# Prix moyen du m2 par commune et type de bien
df_agg = (df_propre.groupBy("nom_commune", "type_local")
                    .agg(round(avg("prix_m2"), 2).alias("prix_m2_moyen")))

print("Prix moyen au m2 par commune et par type de bien : ")
df_agg.orderBy("nom_commune", "type_local").show(10)

print("Plan d'exécution de la requête : ")
df_agg.explain()

Prix moyen au m2 par commune et par type de bien : 
+-------------------+--------------------+-------------+
|        nom_commune|          type_local|prix_m2_moyen|
+-------------------+--------------------+-------------+
|          Abancourt|              Maison|      1858.41|
|             Abscon|         Appartement|       6250.0|
|             Abscon|Local industriel....|      1304.35|
|             Abscon|              Maison|      1182.96|
|      Aix-en-Pévèle|              Maison|      2799.48|
|Allennes-les-Marais|              Maison|      2212.68|
|         Amfroipret|              Maison|      1676.79|
|            Anhiers|              Maison|      1173.17|
|             Aniche|         Appartement|      4743.76|
|             Aniche|Local industriel....|      2729.21|
+-------------------+--------------------+-------------+
only showing top 10 rows
Plan d'exécution de la requête : 
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[nom_commune#

Spark attend un ordre direct pour lancer les calculs. Les commandes comme '.show()' ou '.explain()' sont des "actions". Ces actions forcent Spark à se mettre au travail. (Actions vs transformations) 